In [2]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer
from transformers import AutoModel
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
import evaluate
import numpy as np

In [6]:
raw_dataset = load_dataset("Intel/polite-guard")

In [21]:
sample_size = 500

In [ ]:
# train_valid_test_dataset = DatasetDict({
#     split: raw_dataset[split]
#              .shuffle(seed=42)    # reproducible
#              .select(range(min(sample_size, len(raw_dataset[split]))))
#     for split in ["train","validation","test"]
# })

In [7]:
train_valid_test_dataset = DatasetDict({
    'train': raw_dataset['train'],
    'validation': raw_dataset['validation'],
    'test': raw_dataset['test']
})

label2id = {'polite': 0, 'somewhat polite': 1,'neutral': 2, 'impolite': 3}

In [8]:
def my_preprocess_function(tokenizer):
    def apply(sample):
        toks = tokenizer(sample["text"], truncation=True, padding=True)
        labels = [label2id[l] for l in sample["label"]]
        toks["labels"] = labels
        return toks
    return apply

In [37]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenized_dataset = train_valid_test_dataset.map(
    my_preprocess_function(tokenizer),
    batched=True,
    remove_columns=["text", "source", "reasoning", "label"],
)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels, average="weighted")

In [6]:
training_args = TrainingArguments(
    output_dir="./polite_guard_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch", # run validation at the end of each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [7]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.214400,0.217623,0.914677
2,0.170100,0.209443,0.926776
3,0.132000,0.232471,0.926070


TrainOutput(global_step=15000, training_loss=0.1926816027323405, metrics={'train_runtime': 1623.0155, 'train_samples_per_second': 147.873, 'train_steps_per_second': 9.242, 'total_flos': 1.4594933653507584e+16, 'train_loss': 0.1926816027323405, 'epoch': 3.0})

In [8]:
trainer.evaluate()

{'eval_loss': 0.20944257080554962,
 'eval_f1': 0.9267758190197818,
 'eval_runtime': 13.9607,
 'eval_samples_per_second': 716.298,
 'eval_steps_per_second': 44.769,
 'epoch': 3.0}

In [9]:
trainer.predict(test_dataset=tokenized_dataset["test"])

PredictionOutput(predictions=array([[-3.5345478 , -1.9887767 , -1.8110075 ,  6.3626714 ],
       [-3.1092777 , -2.549419  , -1.4811376 ,  6.3443394 ],
       [-4.1126876 ,  6.4521422 , -2.3847249 , -1.0416496 ],
       ...,
       [-2.8738952 ,  4.6640415 , -0.94364095, -1.4730346 ],
       [-0.51404417, -0.43912917, -2.4369316 ,  2.824226  ],
       [-3.3343275 , -0.3327029 ,  6.187421  , -2.065821  ]],
      shape=(10200, 4), dtype=float32), label_ids=array([3, 3, 1, ..., 1, 2, 3], shape=(10200,)), metrics={'test_loss': 0.27079665660858154, 'test_f1': 0.9165636629366327, 'test_runtime': 13.7155, 'test_samples_per_second': 743.685, 'test_steps_per_second': 46.517})

In [10]:
trainer.save_model()

## PEFT with LoRA

In [38]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the base model, apply LoRA later
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
from peft import LoraConfig, TaskType, get_peft_model

peft_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
)

In [40]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()  # Confirm small amount of trainable parameters

trainable params: 297,988 || all params: 109,783,304 || trainable%: 0.2714


In [41]:
training_args = TrainingArguments(
    output_dir="./results_lora",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch", # run validation at the end of each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    label_names=["labels"],
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [42]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.310100,0.291464,0.886743
2,0.288900,0.266541,0.895353
3,0.285000,0.260133,0.897959


TrainOutput(global_step=15000, training_loss=0.3508183848063151, metrics={'train_runtime': 1045.8201, 'train_samples_per_second': 229.485, 'train_steps_per_second': 14.343, 'total_flos': 1.4645711609699328e+16, 'train_loss': 0.3508183848063151, 'epoch': 3.0})

In [43]:
trainer.evaluate()

{'eval_loss': 0.26013290882110596,
 'eval_f1': 0.897958518069893,
 'eval_runtime': 14.7021,
 'eval_samples_per_second': 680.174,
 'eval_steps_per_second': 42.511,
 'epoch': 3.0}

In [44]:
trainer.predict(test_dataset=tokenized_dataset["test"])

PredictionOutput(predictions=array([[-4.9126515 , -1.461344  ,  1.3464283 ,  3.749132  ],
       [-2.5267847 , -2.0563517 , -0.8228667 ,  4.320741  ],
       [-1.8849846 ,  5.056414  , -1.5040767 , -1.7671883 ],
       ...,
       [-4.1623707 ,  2.4442544 ,  0.86549616,  0.24074401],
       [-1.6163902 , -0.16004431, -1.4702787 ,  2.740533  ],
       [-5.364718  ,  2.8108654 ,  2.5137722 , -1.1157687 ]],
      shape=(10200, 4), dtype=float32), label_ids=array([3, 3, 1, ..., 1, 2, 3], shape=(10200,)), metrics={'test_loss': 0.30074983835220337, 'test_f1': 0.888756809686922, 'test_runtime': 14.4715, 'test_samples_per_second': 704.835, 'test_steps_per_second': 44.087})

In [45]:
trainer.save_model()

In [9]:
roberta_model_name = "roberta-large"
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_model_name)
roberta_model = AutoModelForSequenceClassification.from_pretrained(roberta_model_name, num_labels=4)
roberta_tokenized_dataset = train_valid_test_dataset.map(
    my_preprocess_function(roberta_tokenizer),
    batched=True,
    remove_columns=["text", "source", "reasoning", "label"],
)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/80000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10200 [00:00<?, ? examples/s]

In [13]:
roberta_model_lora = get_peft_model(roberta_model, peft_config)
roberta_model_lora.print_trainable_parameters()  # Confirm small amount of trainable parameters

trainable params: 1,840,132 || all params: 357,203,976 || trainable%: 0.5151


In [15]:
training_args = TrainingArguments(
    output_dir=f"./results_{roberta_model_name}-lora",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch", # run validation at the end of each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    label_names=["labels"],
)

data_collator = DataCollatorWithPadding(tokenizer=roberta_tokenizer)

trainer = Trainer(
    model=roberta_model_lora,
    args=training_args,
    train_dataset=roberta_tokenized_dataset["train"],
    eval_dataset=roberta_tokenized_dataset["validation"],
    processing_class=roberta_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.255000,0.263684,0.899316
2,0.236800,0.246663,0.903024
3,0.234700,0.239802,0.908096


TrainOutput(global_step=15000, training_loss=0.2596146993001302, metrics={'train_runtime': 3140.3402, 'train_samples_per_second': 76.425, 'train_steps_per_second': 4.777, 'total_flos': 4.999461331271117e+16, 'train_loss': 0.2596146993001302, 'epoch': 3.0})

In [17]:
trainer.evaluate()

{'eval_loss': 0.23980239033699036,
 'eval_f1': 0.908096463987454,
 'eval_runtime': 43.8395,
 'eval_samples_per_second': 228.105,
 'eval_steps_per_second': 14.257,
 'epoch': 3.0}

In [18]:
trainer.predict(test_dataset=roberta_tokenized_dataset["test"])

PredictionOutput(predictions=array([[-0.45575258,  6.3291507 , -2.0404243 , -6.4984403 ],
       [-2.3128579 ,  6.765893  , -2.6107435 , -3.7109463 ],
       [-3.6108682 , -1.0510111 ,  7.160719  , -3.3526347 ],
       ...,
       [-3.2616649 ,  1.2849271 ,  1.5459085 , -0.24709669],
       [ 2.6021829 , -0.19929706, -1.1108807 , -2.0636885 ],
       [ 2.4273965 , -0.2999794 ,  1.0766553 , -4.917335  ]],
      shape=(10200, 4), dtype=float32), label_ids=array([1, 1, 2, ..., 2, 0, 1], shape=(10200,)), metrics={'test_loss': 0.2773815095424652, 'test_f1': 0.9027235713754939, 'test_runtime': 42.3289, 'test_samples_per_second': 240.97, 'test_steps_per_second': 15.072})